# 00 — Programs, design, and the pseudobulk cache

Folder question: **what do antigen-specific CAR-T (PSCA vs the non-specific TAG72 CAR), direct
radiation (Tu1) and indirect / abscopal radiation (Tu2) do to MHC, cytotoxic, exhaustion and
chemokine programs — in which cell types — and which of those should be taken into the spatial
SPP1–CD44 analysis?**

This notebook produces no biology. It writes the four things every later notebook reads:

| file | what |
|---|---|
| `pseudobulk_counts.parquet` | summed **raw** counts per (cell group × sample), all genes |
| `pseudobulk_meta.csv` | group, sample, arm, site, mouse, n_cells |
| `cell_panel.parquet` | per-cell raw counts for ~500 panel genes + per-cell metadata |
| `programs.json` | the program catalogue: family, genes, source, ligand / receptor role |

Cell groups are deliberately overlapping (lvl2 types, lvl3 states, SPP1-TAM calls, pooled
lymphocytes); each later notebook says which partition it uses.

### Contrasts every later notebook runs
| name | A vs B | strata |
|---|---|---|
| **PSCA** | PSCA CAR vs non-specific TAG72 CAR | non-RT and RT, each site |
| **direct RT** | RT vs no-RT in Tu1 | per CAR background |
| **indirect RT** | RT vs no-RT in Tu2 (abscopal) | per CAR background |
| **CAR × RT** | (RT effect in PSCA) − (RT effect in TAG72) | per site |
| **within-mouse DiD** | (Tu1 − Tu2)_RT − (Tu1 − Tu2)_non-RT | per background |
| treated vs NoTx | pipeline check only, never a claim | |

The **mouse / sample is the unit everywhere**.

## 1. Config

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, time
from pathlib import Path
import numpy as np, pandas as pd, h5py, anndata as ad
import scipy.sparse as sp
import scanpy as sc

pd.set_option("display.width", 260); pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 300)

_HPC = Path("/coh_labs/yunroseli/Jona/CAR-T")
_LOC = Path("/Users/janzules/Roselab/Spatial/CAR_T")
PROJ = _LOC if _LOC.exists() else _HPC
SC_REF  = (PROJ/"data/sc-reference/annotation_v4.h5ad" if PROJ is _LOC
           else PROJ/"data/zarr/fullDataset/annotating_references/annotation_v4.h5ad")
ZARR    = (PROJ/"data/zarrFiles/1_C2l_annotations_400" if PROJ is _LOC
           else PROJ/"data/zarr/fullDataset/processing_Zarr/1_C2l_annotated_CC")
GMT_DIR = PROJ/"data/references/pathways"
SIG_CSV = PROJ/"results/spp1_analysis/sc_spp1_tam_signature.csv"
ALICE   = PROJ/"results/spp1_analysis/mac_lvl3_lvl4_lvl5_labels.parquet"
OUT = PROJ/"results/mhc_cytotox"; FIG = OUT/"figures"
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

SC2ARM = {"NoTx": "NoTx", "CyPSCA": "CyPSCA", "CyTAG72": "CyT72",
          "RTCyPSCA": "RTCyPSCA", "RTCyTAG72": "RTCyT72"}
ARM_ORDER = ["NoTx", "CyPSCA", "CyT72", "RTCyPSCA", "RTCyT72"]
MIN_CELLS = 10
SEED = 0
for p_, l_ in [(SC_REF, "single cell"), (GMT_DIR, "GMTs"), (SIG_CSV, "SPP1 signature"),
               (ALICE, "Alice lvl4"), (ZARR, "spatial")]:
    print(f"{l_:<15s} {'OK     ' if Path(p_).exists() else 'MISSING'} {p_}")

single cell     OK      /Users/janzules/Roselab/Spatial/CAR_T/data/sc-reference/annotation_v4.h5ad
GMTs            OK      /Users/janzules/Roselab/Spatial/CAR_T/data/references/pathways
SPP1 signature  OK      /Users/janzules/Roselab/Spatial/CAR_T/results/spp1_analysis/sc_spp1_tam_signature.csv
Alice lvl4      OK      /Users/janzules/Roselab/Spatial/CAR_T/results/spp1_analysis/mac_lvl3_lvl4_lvl5_labels.parquet
spatial         OK      /Users/janzules/Roselab/Spatial/CAR_T/data/zarrFiles/1_C2l_annotations_400


## 2. Load, design, and cell labels

`.X` is log-normalised (used only to score the SPP1-TAM signature exactly as `spp1Analysis/09`
did); `layers/counts` is raw and is what every pseudobulk sums.

In [2]:
t0 = time.time()
with h5py.File(SC_REF, "r") as f:
    obs = ad.io.read_elem(f["obs"]); var = ad.io.read_elem(f["var"])
    X = ad.io.read_elem(f["X"]); C = ad.io.read_elem(f["layers/counts"])
genes = pd.Index(var.index.astype(str))
C = sp.csr_matrix(C)
print(f"{C.shape[0]:,} cells x {C.shape[1]:,} genes, read in {time.time()-t0:.0f}s | "
      f"counts integer: {float(C.data[:1000].max()) == int(C.data[:1000].max())}")

o = pd.DataFrame(index=obs.index)
o["sample"] = obs["file_name"].astype(str)
p = o["sample"].str.extract(r"^(?P<sc>.+)_Tu(?P<site>\d)_(?P<rep>\d+)$")
assert p.notna().all().all()
o["arm"] = p["sc"].map(SC2ARM); o["site"] = p["site"].astype(int)
o["mouse"] = p["sc"] + "_m" + p["rep"]
o["ct2"] = obs["cell_type_lvl2"].astype(str).values
o["ct3"] = obs["cell_type_lvl3"].astype(str).values
o["total_counts"] = np.asarray(C.sum(1)).ravel()
o["RT"] = o.arm.str.startswith("RT")
o["cart"] = o.arm.map({"CyPSCA": "PSCA", "RTCyPSCA": "PSCA", "CyT72": "TAG72",
                       "RTCyT72": "TAG72", "NoTx": "none"})
MAC = ["M1_like_Mac", "M2_like_Mac", "Intermediate_Mac"]
o["is_mac"] = o.ct2.isin(MAC)

# SPP1-TAM, definition 1: signature top quartile among macrophages (spp1Analysis/09 TAM_global)
adata = ad.AnnData(X=X, obs=obs[[]].copy(), var=pd.DataFrame(index=genes)); del X
sig = [g for g in pd.read_csv(SIG_CSV)["gene"] if g in genes]
sc.tl.score_genes(adata, sig, score_name="SPP1_TAM_sig", ctrl_size=50, random_state=0)
o["SPP1_TAM_sig"] = adata.obs["SPP1_TAM_sig"].values; del adata
s_ = o.loc[o.is_mac, "SPP1_TAM_sig"]
o["SPP1_TAM"] = o.is_mac & (o.SPP1_TAM_sig >= s_.quantile(0.75))
# SPP1-TAM, definition 2: Alice's macrophage-only annotation (cell_type_lvl4 == Spp1_pos_mac)
lvl4 = pd.read_parquet(ALICE)["cell_type_lvl4"].astype(str)
o["lvl4"] = lvl4.reindex(o.index).fillna("NA").values
o["SPP1_alice"] = o.lvl4 == "Spp1_pos_mac"
print(f"SPP1 signature: {len(sig)} genes | SPP1_TAM (top quartile) {o.SPP1_TAM.sum():,} | "
      f"Spp1_pos_mac (Alice) {o.SPP1_alice.sum():,} of {o.is_mac.sum():,} macrophages")
print("\nSPP1_TAM cells by lvl3 state:")
print(o[o.SPP1_TAM].ct3.value_counts().to_string())

design = (o.groupby("sample").agg(arm=("arm", "first"), site=("site", "first"),
                                  mouse=("mouse", "first"), n_cells=("arm", "size"),
                                  RT=("RT", "first"), cart=("cart", "first")).reset_index())
design.to_csv(OUT/"design_samples.csv", index=False)
print("\nsamples per arm x site:")
print(design.pivot_table(index="arm", columns="site", values="sample", aggfunc="count")
        .reindex(ARM_ORDER).to_string())

93,548 cells x 22,949 genes, read in 2s | counts integer: True


SPP1 signature: 50 genes | SPP1_TAM (top quartile) 12,119 | Spp1_pos_mac (Alice) 29,582 of 48,475 macrophages

SPP1_TAM cells by lvl3 state:
ct3
Apoe_high_Mac         9834
Arg1_high_Mac         1161
Fn1_high_Mac          1108
IFN_stimulated_Mac      13
Mmp12_high_Mac           3

samples per arm x site:
site      1  2
arm           
NoTx      3  3
CyPSCA    3  3
CyT72     2  3
RTCyPSCA  3  3
RTCyT72   3  3


## 3. Cell groups and the pseudobulk cache

* every `cell_type_lvl2` type (Erythrocyte excluded) — the **partition** used for contribution
  shares in `01`;
* the lvl3 states that matter here (macrophage, DC, NK and proliferating CD8 states);
* `mac_all`, both SPP1-TAM calls, and the exclusive pairs `04b` compares
  (`SPP1TAM_notIFN` vs `IFNmac_notSPP1`, and the Alice-based equivalents);
* pooled lymphocytes: `cytotoxic_lymph` = CD8_T + NKT + NK, and `T_all` = CD8 + CD4 + Treg + NKT.

In [3]:
G = {}
for c in sorted(o.ct2.unique()):
    if c != "Erythrocyte": G[c] = o.ct2.values == c
for c in ["IFN_stimulated_Mac", "Apoe_high_Mac", "Arg1_high_Mac", "Fn1_high_Mac", "Mmp12_high_Mac",
          "IFN_stimulated_Mono", "Phagocytic_Mono", "MHCII_high_DC", "Low_migratory_DC",
          "Fscn1_high_DC", "Cytotoxic_NK", "Granzyme_high_NK", "Proliferating_NK", "Eomes_low_NK",
          "Proliferating_CD8_T"]:
    G[c] = o.ct3.values == c
ifn = o.ct3.values == "IFN_stimulated_Mac"
G["mac_all"] = o.is_mac.values
G["SPP1_TAM"] = o.SPP1_TAM.values
G["SPP1_alice"] = o.SPP1_alice.values
G["SPP1TAM_notIFN"] = o.SPP1_TAM.values & ~ifn
G["IFNmac_notSPP1"] = ifn & ~o.SPP1_TAM.values
G["SPP1alice_notIFN"] = o.SPP1_alice.values & ~ifn
G["IFNmac_SPP1neg_alice"] = ifn & (o.lvl4.values == "Spp1_neg_mac")
G["cytotoxic_lymph"] = o.ct2.isin(["CD8_T", "NKT", "NK"]).values
G["T_all"] = o.ct2.isin(["CD8_T", "CD4_T", "Treg", "NKT"]).values

samples = design["sample"].tolist()
smp_idx = pd.Series(np.arange(len(samples)), index=samples)[o["sample"]].values
rows, cols, keys = [], [], []
for gname, m in G.items():
    for s_i, s in enumerate(samples):
        sel = np.where(m & (smp_idx == s_i))[0]
        if len(sel) == 0: continue
        r = len(keys); keys.append((gname, s, len(sel)))
        rows.append(np.full(len(sel), r)); cols.append(sel)
M = sp.csr_matrix((np.ones(sum(len(x) for x in rows)), (np.concatenate(rows), np.concatenate(cols))),
                  shape=(len(keys), C.shape[0]))
t0 = time.time()
PB = (M @ C).toarray().astype(np.int64)
print(f"pseudobulk: {PB.shape[0]} (group x sample) rows x {PB.shape[1]:,} genes in {time.time()-t0:.0f}s")

PBM = pd.DataFrame(keys, columns=["group", "sample", "n_cells"]).merge(design.drop(columns="n_cells"),
                                                                       on="sample")
PBM["key"] = PBM.group + "|" + PBM["sample"]
pd.DataFrame(PB, index=PBM.key, columns=genes).to_parquet(OUT/"pseudobulk_counts.parquet")
PBM.to_csv(OUT/"pseudobulk_meta.csv", index=False)

grid = PBM.pivot_table(index="group", columns="sample", values="n_cells", aggfunc="sum").fillna(0)
grid = grid[samples]
use = (grid >= MIN_CELLS).sum(1)
print(f"\nusable samples (>= {MIN_CELLS} cells) per group (of 29; 23 treated):")
print(pd.DataFrame({"usable_samples": use,
                    "usable_treated": (grid[[s for s in samples if not s.startswith("NoTx")]]
                                       >= MIN_CELLS).sum(1),
                    "median_cells": grid.median(1), "total_cells": grid.sum(1)})
        .sort_values("total_cells", ascending=False).astype(int).to_string())
grid.to_csv(OUT/"cells_per_group_sample.csv")

pseudobulk: 1158 (group x sample) rows x 22,949 genes in 2s



usable samples (>= 10 cells) per group (of 29; 23 treated):
                      usable_samples  usable_treated  median_cells  total_cells
group                                                                          
mac_all                           29              23          1861        48475
SPP1_alice                        29              23          1095        29582
Classical_Mono                    29              23          1042        25955
M2_like_Mac                       29              23           704        23538
SPP1alice_notIFN                  29              23           780        23000
IFN_stimulated_Mac                29              23           479        15716
M1_like_Mac                       29              23           479        15716
IFNmac_notSPP1                    29              23           476        15703
IFN_stimulated_Mono               24              23           496        13616
Phagocytic_Mono                   29              23       

## 4. The program catalogue

Families: **MHC-I, MHC-II, IFN drivers, cytotoxicity, exhaustion, checkpoint ligands, chemokines,
controls.** Curated modules are written out; GMT sets are referenced by exact name.

**Exhaustion** has no set in these GMTs, so it is curated from the CD8 exhaustion literature
(inhibitory receptors and TOX/NR4A-family TFs — Wherry & Kurachi 2015 *Nat Rev Immunol*; Khan et
al. 2019 *Nature*; progenitor vs terminal exhausted — Miller et al. 2019 *Nat Immunol*; Beltra et
al. 2020 *Immunity*).

**Chemokines** are kept as separate **ligand** and **receptor** panels per axis, never one
combined score.

**Sensitivity variants** — `cyto_effector_noCcl5` and `exh_progenitor_noXcl1` — because Ccl5 and
Xcl1 are themselves chemokines, and leaving them in would make "cytotoxic" and "chemokine"
partly the same measurement.

In [4]:
def read_gmt(p_):
    out = {}
    for line in open(p_):
        x = line.rstrip("\n").split("\t")
        if len(x) >= 3: out[x[0]] = x[2:]
    return out


GMT = {}
for f_ in sorted(GMT_DIR.glob("*.gmt")): GMT.update(read_gmt(f_))

CUR = {   # family, genes
    "MHC_I_machinery":   ("MHC-I", ["H2-K1", "H2-D1", "H2-Q4", "H2-Q7", "H2-T23", "B2m", "Tap1",
                                    "Tap2", "Tapbp", "Psmb8", "Psmb9", "Psmb10", "Nlrc5", "Erap1"]),
    "MHC_II_core":       ("MHC-II", ["H2-Aa", "H2-Ab1", "H2-Eb1", "H2-DMa", "H2-DMb1", "Cd74", "Ciita"]),
    "ISG_core":          ("IFN", ["Irf1", "Irf7", "Stat1", "Isg15", "Ifit1", "Ifit3", "Oasl1",
                                  "Bst2", "Gbp2", "Gbp5"]),
    "cyto_effector":     ("cytotoxicity", ["Gzma", "Gzmb", "Gzmk", "Prf1", "Nkg7", "Ifng", "Fasl",
                                           "Ctsw", "Ccl5", "Klrk1", "Klrd1"]),
    "cyto_effector_noCcl5": ("cytotoxicity", ["Gzma", "Gzmb", "Gzmk", "Prf1", "Nkg7", "Ifng", "Fasl",
                                              "Ctsw", "Klrk1", "Klrd1"]),
    "exh_inhibitory_receptors": ("exhaustion", ["Pdcd1", "Havcr2", "Lag3", "Tigit", "Ctla4",
                                                "Cd160", "Cd244a", "Entpd1"]),
    "exh_TFs":           ("exhaustion", ["Tox", "Tox2", "Nr4a1", "Nr4a2", "Nr4a3", "Batf", "Irf4"]),
    "exh_progenitor":    ("exhaustion", ["Tcf7", "Slamf6", "Xcl1", "Id3", "Cxcr5"]),
    "exh_progenitor_noXcl1": ("exhaustion", ["Tcf7", "Slamf6", "Id3", "Cxcr5"]),
    "exh_terminal":      ("exhaustion", ["Havcr2", "Entpd1", "Cd38", "Cd101", "Gzmb"]),
    "checkpoint_ligands": ("checkpoint", ["Cd274", "Pdcd1lg2", "Lgals9", "Pvr", "Nectin2", "Cd80",
                                          "Cd86", "Ceacam1"]),
}
FROM_GMT = {
    "MHC_I_Reactome":       ("MHC-I", "REACTOME_CLASS_I_MHC_MEDIATED_ANTIGEN_PROCESSING_PRESENTATION"),
    "MHC_I_loading_Reactome": ("MHC-I", "REACTOME_ANTIGEN_PRESENTATION_FOLDING_ASSEMBLY_AND_PEPTIDE_LOADING_OF_CLASS_I_MHC"),
    "MHC_I_GO":             ("MHC-I", "GOBP_ANTIGEN_PROCESSING_AND_PRESENTATION_OF_PEPTIDE_ANTIGEN_VIA_MHC_CLASS_I"),
    "MHC_II_Reactome":      ("MHC-II", "REACTOME_MHC_CLASS_II_ANTIGEN_PRESENTATION"),
    "MHC_II_GO":            ("MHC-II", "GOBP_ANTIGEN_PROCESSING_AND_PRESENTATION_OF_PEPTIDE_OR_POLYSACCHARIDE_ANTIGEN_VIA_MHC_CLASS_II"),
    "IFNG_Hallmark":        ("IFN", "HALLMARK_INTERFERON_GAMMA_RESPONSE"),
    "IFNA_Hallmark":        ("IFN", "HALLMARK_INTERFERON_ALPHA_RESPONSE"),
    "IFNG_Reactome":        ("IFN", "REACTOME_INTERFERON_GAMMA_SIGNALING"),
    "R01_granzyme_GO":      ("cytotoxicity", "GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY"),
    "T_cytotox_GO":         ("cytotoxicity", "GOBP_T_CELL_MEDIATED_CYTOTOXICITY"),
    "leuk_cytotox_GO":      ("cytotoxicity", "GOBP_LEUKOCYTE_MEDIATED_CYTOTOXICITY"),
    "cell_killing_GO":      ("cytotoxicity", "GOBP_CELL_KILLING"),
    "NK_vs_tumour_GO":      ("cytotoxicity", "GOBP_NATURAL_KILLER_CELL_MEDIATED_IMMUNE_RESPONSE_TO_TUMOR_CELL"),
    "Tcytotoxic_Biocarta":  ("cytotoxicity", "BIOCARTA_TCYTOTOXIC_PATHWAY"),
    "allograft_Hallmark":   ("cytotoxicity", "HALLMARK_ALLOGRAFT_REJECTION"),
    "PD1_coinhibition_Reactome": ("exhaustion", "REACTOME_CO_INHIBITION_BY_PD_1"),
    "anergy_GO":            ("exhaustion", "GOBP_LYMPHOCYTE_ANERGY"),
    "chemokine_receptors_bind_Reactome": ("chemokine-pathway", "REACTOME_CHEMOKINE_RECEPTORS_BIND_CHEMOKINES"),
    "chemokine_signaling_WP": ("chemokine-pathway", "WP_CHEMOKINE_SIGNALING_PATHWAY"),
    "response_to_chemokine_GO": ("chemokine-pathway", "GOBP_RESPONSE_TO_CHEMOKINE"),
    "chemokine_production_GO": ("chemokine-pathway", "GOBP_CHEMOKINE_PRODUCTION"),
    "leukocyte_chemotaxis_GO": ("chemokine-pathway", "GOBP_LEUKOCYTE_CHEMOTAXIS"),
    "lymphocyte_chemotaxis_GO": ("chemokine-pathway", "GOBP_LYMPHOCYTE_CHEMOTAXIS"),
    "T_cell_chemotaxis_GO": ("chemokine-pathway", "GOBP_T_CELL_CHEMOTAXIS"),
    "NK_chemotaxis_GO":     ("chemokine-pathway", "GOBP_NATURAL_KILLER_CELL_CHEMOTAXIS"),
    "T_cell_migration_GO":  ("chemokine-pathway", "GOBP_T_CELL_MIGRATION"),
    "lymphocyte_migration_GO": ("chemokine-pathway", "GOBP_LYMPHOCYTE_MIGRATION"),
    "monocyte_chemotaxis_GO": ("chemokine-pathway", "GOBP_MONOCYTE_CHEMOTAXIS"),
    "macrophage_chemotaxis_GO": ("chemokine-pathway", "GOBP_MACROPHAGE_CHEMOTAXIS"),
    "myeloid_migration_GO": ("chemokine-pathway", "GOBP_MYELOID_LEUKOCYTE_MIGRATION"),
    "neutrophil_chemotaxis_GO": ("chemokine-pathway", "GOBP_NEUTROPHIL_CHEMOTAXIS"),
    "granulocyte_chemotaxis_GO": ("chemokine-pathway", "GOBP_GRANULOCYTE_CHEMOTAXIS"),
    "DC_chemotaxis_GO":     ("chemokine-pathway", "GOBP_DENDRITIC_CELL_CHEMOTAXIS"),
    "CXCR4_signaling_GO":   ("chemokine-pathway", "GOBP_C_X_C_CHEMOKINE_RECEPTOR_CXCR4_SIGNALING_PATHWAY"),
    "CCR5_Biocarta":        ("chemokine-pathway", "BIOCARTA_CCR5_PATHWAY"),
    "CXCR4_Biocarta":       ("chemokine-pathway", "BIOCARTA_CXCR4_PATHWAY"),
    "OXPHOS_Hallmark":      ("control", "HALLMARK_OXIDATIVE_PHOSPHORYLATION"),
    "MYC_Hallmark":         ("control", "HALLMARK_MYC_TARGETS_V1"),
}
# chemokine axes: (priority, ligands, receptors, the receiving populations the axis is about)
AXES = {
    "CXCR3": ("primary",   ["Cxcl9", "Cxcl10", "Cxcl11"], ["Cxcr3"], ["CD8_T", "NK", "Treg", "NKT"]),
    "XCR1":  ("primary",   ["Xcl1"], ["Xcr1"], ["cDC"]),
    "CCR5":  ("primary",   ["Ccl3", "Ccl4", "Ccl5"], ["Ccr5"], ["NK", "CD8_T", "M1_like_Mac", "M2_like_Mac"]),
    "CCR2":  ("primary",   ["Ccl2", "Ccl7", "Ccl12"], ["Ccr2"], ["Classical_Mono", "Intermediate_Mac", "NK"]),
    "CXCR2": ("primary",   ["Cxcl1", "Cxcl2", "Cxcl3", "Cxcl5", "Ppbp"], ["Cxcr2"], ["N1_like_Neu", "N2_like_Neu"]),
    "CCR4":  ("primary",   ["Ccl17", "Ccl22"], ["Ccr4"], ["Treg", "CD4_T", "NKT"]),
    "CXCR6": ("secondary", ["Cxcl16"], ["Cxcr6"], ["CD8_T", "NKT", "CD4_T"]),
    "CXCR4": ("secondary", ["Cxcl12"], ["Cxcr4", "Ackr3"], ["CD8_T", "NK", "M2_like_Mac", "Classical_Mono"]),
}

# --- Cxcl11: strain validity and presence in the data -----------------------------------------
import zarr
sp_var = set(ad.io.read_elem(zarr.open_group(str(ZARR/"tables/segmentation_counts"), mode="r")["var"])
             .index.astype(str)) if Path(ZARR).exists() else set()
print("Cxcl11 check")
print(f"  in single-cell var_names: {'Cxcl11' in genes} | in spatial var_names: {'Cxcl11' in sp_var}")
print("  C57BL/6 mice carry a non-functional Cxcl11 allele; this model (Myc-CaP) is FVB-derived, so")
print("  strain would not by itself exclude it. It is excluded here because the gene is not in the")
print("  reference annotation at all -> the CXCR3 ligand panel is Cxcl9 + Cxcl10.")
for k, (pri, L, R, tg) in AXES.items():
    AXES[k] = (pri, [g for g in L if g in genes], [g for g in R if g in genes], tg)

PROGRAMS = {}
for k, (fam, gl) in CUR.items():
    PROGRAMS[k] = dict(family=fam, source="curated", role="program",
                       genes=[g for g in dict.fromkeys(gl) if g in genes], n_in_set=len(set(gl)))
for k, (fam, gs) in FROM_GMT.items():
    gl = GMT.get(gs, [])
    if not gl: print(f"  [missing from GMT] {gs}"); continue
    PROGRAMS[k] = dict(family=fam, source=f"GMT:{gs}", role="program",
                       genes=[g for g in dict.fromkeys(gl) if g in genes], n_in_set=len(set(gl)))
for k, (pri, L, R, tg) in AXES.items():
    PROGRAMS[f"{k}_ligands"] = dict(family="chemokine", source="curated", role="ligand", axis=k,
                                    priority=pri, genes=L, n_in_set=len(L), targets=tg)
    PROGRAMS[f"{k}_receptors"] = dict(family="chemokine", source="curated", role="receptor", axis=k,
                                      priority=pri, genes=R, n_in_set=len(R), targets=tg)
json.dump(dict(programs=PROGRAMS, axes={k: dict(priority=v[0], ligands=v[1], receptors=v[2],
                                                 targets=v[3]) for k, v in AXES.items()}),
          open(OUT/"programs.json", "w"), indent=1)
print(f"\n{len(PROGRAMS)} programs written to programs.json")

Cxcl11 check
  in single-cell var_names: False | in spatial var_names: False
  C57BL/6 mice carry a non-functional Cxcl11 allele; this model (Myc-CaP) is FVB-derived, so
  strain would not by itself exclude it. It is excluded here because the gene is not in the
  reference annotation at all -> the CXCR3 ligand panel is Cxcl9 + Cxcl10.

65 programs written to programs.json


## 5. The per-cell panel table

Raw counts for every gene of the curated modules, chemokine panel, Hallmark IFN-γ, the R01
granzyme set, the SPP1 signature, ambient markers and lineage markers — for the cell-level
analyses (`04` exhaustion GLMMs, `04b` receptor-positive fractions and within-sample
correlations, `06` score audit).

In [5]:
AMBIENT = ["C1qa", "C1qb", "C1qc", "Ms4a7", "Adgre1", "Csf1r"]
LINEAGE = ["Cd8a", "Cd8b1", "Cd3e", "Cd4", "Foxp3", "Ncr1", "Klrb1c", "Spp1", "Cd44", "Ptprc",
           "Lyz2", "Epcam", "Krt8", "Krt18"]
panel = set(AMBIENT + LINEAGE + sig)
for k, v in PROGRAMS.items():
    if v["source"] == "curated" or k in ("IFNG_Hallmark", "MHC_II_Reactome", "R01_granzyme_GO",
                                         "PD1_coinhibition_Reactome", "MHC_I_loading_Reactome"):
        panel |= set(v["genes"])
panel = [g for g in genes if g in panel]
gi = genes.get_indexer(panel)
cp = pd.DataFrame(C[:, gi].toarray().astype(np.int32), index=o.index, columns=panel)
cell_meta = o[["sample", "arm", "site", "mouse", "RT", "cart", "ct2", "ct3", "lvl4", "is_mac",
               "SPP1_TAM", "SPP1_alice", "SPP1_TAM_sig", "total_counts"]]
cell_meta.join(cp).to_parquet(OUT/"cell_panel.parquet")
print(f"cell_panel.parquet: {cp.shape[0]:,} cells x {cp.shape[1]} panel genes")

cell_panel.parquet: 93,548 cells x 519 panel genes


## 6. Coverage — which programs can each cell type actually read?

For each program: genes in the set, genes in the object, and in each major cell type the share
of program genes detected in >= 10% of its cells. A program with few detectable genes in a cell
type is **underpowered there**, and a null in that cell type is not "no effect".

In [6]:
CT_SHOW = ["Cancer_cell", "M1_like_Mac", "M2_like_Mac", "Intermediate_Mac", "Classical_Mono",
           "N1_like_Neu", "N2_like_Neu", "cDC", "CD8_T", "NK", "NKT", "CD4_T", "Treg",
           "Fibroblast", "Endothelial"]
det = {}
for c in CT_SHOW:
    m = o.ct2.values == c
    det[c] = pd.Series(np.asarray((C[m] > 0).mean(0)).ravel(), index=genes)
det = pd.DataFrame(det)
det.loc[[g for g in panel]].to_csv(OUT/"panel_gene_detection_by_celltype.csv")
cov = []
for k, v in PROGRAMS.items():
    gl = v["genes"]
    r = dict(program=k, family=v["family"], role=v["role"], in_set=v["n_in_set"], in_object=len(gl))
    for c in CT_SHOW:
        r[c] = float((det.loc[gl, c] >= 0.10).mean()) if gl else np.nan
    cov.append(r)
COV = pd.DataFrame(cov).set_index("program")
COV.to_csv(OUT/"program_coverage.csv")
print("share of program genes detected in >= 10% of cells, per cell type")
print(COV.round(2).to_string())

share of program genes detected in >= 10% of cells, per cell type
                                              family      role  in_set  in_object  Cancer_cell  M1_like_Mac  M2_like_Mac  Intermediate_Mac  Classical_Mono  N1_like_Neu  N2_like_Neu   cDC  CD8_T    NK   NKT  CD4_T  Treg  Fibroblast  Endothelial
program                                                                                                                                                                                                                                            
MHC_I_machinery                                MHC-I   program      14         14         0.29         1.00         1.00              1.00            1.00         0.86         0.21  1.00   1.00  1.00  1.00   1.00  1.00        1.00         1.00
MHC_II_core                                   MHC-II   program       7          7         0.43         1.00         1.00              1.00            1.00         0.14         0.00  0.86   0.57  0.57  0

## 7. Summary

In [7]:
print("=" * 100); print("SUMMARY — report these back"); print("=" * 100)
print(f"\n[data] {len(o):,} cells, {len(design)} samples, {design.mouse.nunique()} mice; "
      f"{len(G)} cell groups -> {len(PBM)} pseudobulks")
print(f"[SPP1-TAM] top-quartile signature {o.SPP1_TAM.sum():,} | Alice Spp1_pos_mac {o.SPP1_alice.sum():,}"
      f" | overlap with IFN_stimulated_Mac: {int((o.SPP1_TAM & ifn).sum()):,} / "
      f"{int((o.SPP1_alice & ifn).sum()):,}")
print("\n[usable treated samples (>= 10 cells) for the populations this folder is about]")
for gname in ["Cancer_cell", "IFN_stimulated_Mac", "M2_like_Mac", "Classical_Mono", "cDC", "N1_like_Neu",
              "CD8_T", "NK", "NKT", "CD4_T", "Treg", "cytotoxic_lymph", "T_all", "Fibroblast",
              "Endothelial", "SPP1TAM_notIFN", "IFNmac_notSPP1"]:
    tr = [s for s in samples if not s.startswith("NoTx")]
    print(f"    {gname:<20s} {int((grid.loc[gname, tr] >= MIN_CELLS).sum()):>2d}/23   "
          f"median {int(grid.loc[gname].median()):>4d} cells/sample")
print("\n[chemokine axes after the presence check]")
for k, (pri, L, R, tg) in AXES.items():
    rdet = {f"{r_} in {c}": f"{100*det.loc[r_, c]:.0f}%" for r_ in R for c in tg if c in det}
    print(f"    {k:<6s} ({pri:<9s}) ligands {L}  receptors {R}  receptor detection in targets {rdet}")
print("    Cxcl11 excluded: not in the single-cell reference (strain would not exclude it — FVB model).")
print(f"\n[programs] {len(PROGRAMS)} | by family: {pd.Series({k: v['family'] for k, v in PROGRAMS.items()}).value_counts().to_dict()}")
print(f"[files] {OUT}: pseudobulk_counts.parquet, pseudobulk_meta.csv, cell_panel.parquet, programs.json,"
      f"\n        design_samples.csv, cells_per_group_sample.csv, program_coverage.csv, "
      f"panel_gene_detection_by_celltype.csv")
print("=" * 100)

SUMMARY — report these back

[data] 93,548 cells, 29 samples, 15 mice; 42 cell groups -> 1158 pseudobulks
[SPP1-TAM] top-quartile signature 12,119 | Alice Spp1_pos_mac 29,582 | overlap with IFN_stimulated_Mac: 13 / 6,582

[usable treated samples (>= 10 cells) for the populations this folder is about]
    Cancer_cell          23/23   median  175 cells/sample
    IFN_stimulated_Mac   23/23   median  479 cells/sample
    M2_like_Mac          23/23   median  704 cells/sample
    Classical_Mono       23/23   median 1042 cells/sample
    cDC                  23/23   median   45 cells/sample
    N1_like_Neu          22/23   median   58 cells/sample
    CD8_T                16/23   median   14 cells/sample
    NK                   20/23   median   32 cells/sample
    NKT                   4/23   median    2 cells/sample
    CD4_T                 1/23   median    2 cells/sample
    Treg                  0/23   median    2 cells/sample
    cytotoxic_lymph      23/23   median   49 cells/sample
  